In [11]:

import pyvi
from pyvi import ViTokenizer 
import numpy as np 
import pandas as pd 
import stop_words
import re 
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
import json

# Hàm sử dụng để chuẩn hoá chuỗi

In [12]:

def standarlize_duplicate_token(tokens):
    length = len(tokens)
    for i in range(0, length - 2):
        for j in range(i + 1, length - 1):
            if tokens[i].lower() == tokens[j].lower():
                tokens[j] = tokens[i]
    return tokens

def remove_links(text):
    text = re.sub(r'https?:\/\/(www\.)?[-a-zA-Z0–9@:%._\+~#=]{2,256}\.[a-z]{2,6}\b([-a-zA-Z0–9@:%_\+.~#?&//=]*)', '', text, flags=re.MULTILINE)
    text = re.sub(r'[-a-zA-Z0–9@:%._\+~#=]{2,256}\.[a-z]{2,6}\b([-a-zA-Z0–9@:%_\+.~#?&//=]*)', '', text, flags=re.MULTILINE)
    return text


def remove_img(text):
    text = re.sub(r'img_[0-9a-fA-F-]+', '', text, flags=re.MULTILINE)
    return text


def remove_html_tags(text):
    clean = re.compile('<.*?>')
    return re.sub(clean, '', text)


# Hàm tách từ

In [13]:
def vi_term_tokenize(text):
    tokens = [] 
    text = remove_html_tags(text)
   
    
    terms = ViTokenizer.tokenize(text) 
    for term in terms.split(" "): 
        if (term.lower() not in stop_words.STOP_WORDS):
            if ("_" in term) or (term.isalpha() == True) and (len(term) >= 3):
                tokens.append(term) 
    tokens = standarlize_duplicate_token(tokens) 
    return tokens

# Lấy dữ liệu trực tuyến từ trang cafebiz.vn

In [14]:
import requests
import os
from datetime import date, timedelta, datetime
import datetime as dt
from bs4 import BeautifulSoup

In [15]:

def unix_time_to_str(timestamp):
    return datetime.fromtimestamp(int(timestamp)).strftime('%Y-%m-%d %H:%M:%S')


In [26]:

def get_link_like_share_count(link):
    url = "https://sharefb.cnnd.vn/?urls=" + link
    payload = {}
    headers = {
      'Accept': 'application/json, text/javascript, */*; q=0.01',
      'Origin': 'https://cafebiz.vn'
    }

    response = requests.request("GET", url, headers=headers, data = payload)

    return json.loads(response.text)

In [27]:

def get_link_view(news_id):
    url = "https://cmsanalytics.admicro.vn/cms/cafebiz.vn/item/report?news_ids=[" + news_id + "]&cmskey=97492f372c754b059190066f7a75e093&fbclid=IwAR3MEPSL4ubz594m3YwFZU706BIY9vS4RD_AUsFwdK6mTyq4FEhi9-FTDkY"
    payload = {}
    headers = {
      'Accept': 'application/json, text/javascript, */*; q=0.01',
      'Origin': 'https://cmsanalytics.admicro.vn/'
    }

    response = requests.request("GET", url, headers=headers, data = payload)

    return json.loads(response.text)

In [28]:

def get_link_content(link):
    try:
        request = requests.get(link)
    except:
        pass
    
    request_html = BeautifulSoup(request.content, "html.parser")
    
    description = request_html.find("h2", attrs={"class":"sapo"}).get_text().strip()
   
    content = [x.get_text().strip() for x in request_html.find("div", attrs={"class": "detail-content"}).find_all('p')]
   
    tags = [x.text.strip() for x in request_html.find("span", {"class": "tags-item"}).find_all('a')]

    return {
        "description" : description,
        "content" : " ".join(content),
        "tags" : ";".join(tags)
    }

In [29]:

def get_100_news_post():
    url = 'https://nspapi.aiservice.vn/request/client?guid=1509518141984318107&domain=CafeBiz&boxid=4&url=http://cafebiz.vn&numnews=100'
    response = requests.get(url)
    return json.loads(response.text.replace('CafeBiz_Box_4=', ''))

In [30]:

_file_open_mod = 'w' 
_header = True

_data_file_path = os.path.join(os.curdir, 'data.csv')
print("Đang lấy dữ liệu ...")

news_data = pd.DataFrame(columns=['id', 'url', 'title', 'description', 'content', 'tags', 'crawl_date', 'public_date', 'page_view', 'fb_like', 'fb_share', 'fb_comment', 'fb_total'])

news_posts = get_100_news_post()

for post in news_posts['listnews']:
   
    fb = get_link_like_share_count(post['url'])
    view = get_link_view(post['id'])
    content = get_link_content(post['url'])
    news_data.loc[len(news_data)] = [post['id'], post['url'], post['title'], content['description'], content['content'], content['tags'], datetime.now().strftime("%d-%m-%Y  %H:%M:%S"), unix_time_to_str(post['publishDate']), view[0]['total_view']['view_pc'] + view[0]['total_view']['view_mob'], fb[0]['like_count'], fb[0]['share_count'], fb[0]['comment_count'], fb[0]['total_count']];
print("Ghi tập tin ...")
with open(_data_file_path, mode=_file_open_mod, encoding="utf-8") as data_file:
    news_data.to_csv(data_file, header=_header, index=False, line_terminator='\n')
print("Done cralw !")


Đang lấy dữ liệu ...
Ghi tập tin ...
Done cralw !


# Đọc dữ liệu đã lấy về vào Frame dữ liệu

In [31]:
data = pd.read_csv('data.csv')

In [32]:
filter = data.content.str.contains("Tết")
data[filter]

,id,url,title,description,content,tags,crawl_date,public_date,page_view,fb_like,fb_share,fb_comment,fb_total
2,20200109170155502,http://cafebiz.vn/dao-quat-hoa-ngap-tran-pho-t...,"Đào, quất, hoa ngập tràn phố trước rằm tháng Chạp","Rằm tháng chạp, 9/1/2020, không khí mua hoa ch...","Trên chợ hoa Quảng Bá,Tây Hồ đỏ rực cả một con...",Chợ hoa Quảng Bá;người trồng đào;dịp tết nguyê...,15-01-2020 16:05:05,2020-01-09 17:01:07,454,0,0,0,0
6,20200113094356767,http://cafebiz.vn/me-30-con-mc-minh-trang-va-n...,Mẹ 4 con MC Minh Trang và nỗi lo ngày Tết quá ...,"Tết càng hiện đại, nhiều phụ huynh càng nới lỏ...","Thời điểm cận Tết, nhà nhà người người nô nức ...",ăn đủ chất;không khí Tết;đêm giao thừa,15-01-2020 16:05:13,2020-01-13 14:14:00,2927,0,0,0,0
9,20200110003946627,http://cafebiz.vn/da-gay-phan-no-vi-sau-tet-mo...,"Đã gây phẫn nộ vì sau Tết mới trả lương, sếp c...","Xin chia buồn với nhân viên của công ty, thế l...",Tết Nguyên đán đang đến rất gần với chị em côn...,tết nguyên đán;chị em công sở;trời se lạnh;khô...,15-01-2020 16:05:47,2020-01-10 00:39:00,2766,0,0,0,0
15,20200115110423166,http://cafebiz.vn/can-biet-thu-vua-xay-xong-na...,"Căn biệt thự vừa xây xong nằm ""bật ngửa"" giữa ...",Đâu ai ngờ giữa một làng hoa thanh bình lại có...,Sa Đéc vốn là một trong những thành phố nổi ti...,biệt thự;Sa Đéc,15-01-2020 16:05:55,2020-01-15 11:03:51,3772,0,0,0,0
20,20200111150906549,http://cafebiz.vn/anh-thu-phu-la-dong-lon-nhat...,Ảnh: “Thủ phủ” lá dong lớn nhất Hà Nội hối hả ...,"Chỉ cần nhắc đến thương hiệu ""lá dong quê mình...","Cứ vào độ tháng 12 âm lịch hàng năm, người dân...",dịp tết nguyên đán;tết nguyên đán;tết cổ truyề...,15-01-2020 16:06:02,2020-01-11 15:16:00,770,0,0,0,0
26,20200114152829633,http://cafebiz.vn/thuong-tet-bat-dong-san-noi-...,Thưởng Tết bất động sản: Nơi thưởng vài chục t...,"Không ít chủ đầu tư, công ty môi giới địa ốc c...","Lại một cái Tết buồn, lương còn chưa có nói g...",bất động sản;thưởng tết,15-01-2020 16:06:08,2020-01-14 15:28:40,188,0,0,0,0
30,20200115080553635,http://cafebiz.vn/lang-nuoi-ca-chep-do-tat-bat...,Làng nuôi cá chép đỏ tất bật trước ngày cúng ô...,"Chỉ còn ít ngày nữa đến ngày Tết ông Công, ông...","Chỉ còn ít ngày nữa đến ngày Tết ông Công, ôn...",23 tháng chạp;người nuôi cá;tỉnh Phú Thọ;phát ...,15-01-2020 16:06:43,2020-01-15 08:10:15,239,0,0,0,0
32,20200109154624424,http://cafebiz.vn/bat-mi-15-dong-tac-the-duc-g...,"Bật mí 15 động tác thể dục giúp giảm mỡ bụng ""...",Tết đã đến gần mà bạn chưa có bài tập nào để l...,Bây giờ là thời điểm mà mỗi người đều rất cần ...,động tác thể dục;Vóc dáng đẹp;bài tập giảm mỡ ...,15-01-2020 16:06:44,2020-01-09 19:06:00,7242,0,0,0,0
39,20200115101309525,http://cafebiz.vn/tet-nay-ha-noi-giam-so-luong...,Tết này Hà Nội giảm số lượng 'ông đồ',Trung tâm Hoạt động Văn hóa khoa học Văn Miếu ...,Công tác khảo tuyển người viết chữ tại Hội chữ...,Văn Miếu - Quốc Tử Giám;thí sinh tham gia;tôn ...,15-01-2020 16:06:55,2020-01-15 10:12:38,261,0,0,0,0
49,20200115092744838,http://cafebiz.vn/ve-thu-phu-cung-cap-la-dong-...,"Về ""thủ phủ"" cung cấp lá dong gói bánh chưng T...","Gần Tết Nguyên đán, thủ phủ trồng lá dong lại ...","Làng Vĩnh Phúc (xã Quang Vĩnh, Đức Thọ, Hà Tĩn...",hương vị tết;gói bánh chưng;lá dong;Tết,15-01-2020 16:07:07,2020-01-15 09:45:00,456,0,0,0,0


In [33]:

def compute_tf(word_dict, l):
    tf = {}
    sum_nk = len(l)
    for word, count in word_dict.items():
        tf[word] = count/sum_nk
    return tf

In [34]:

import string
from math import log

def compute_idf(docs_vector):
    n = len(docs_vector)
    idf = dict.fromkeys(docs_vector[0].keys(), 0) 
    for document in docs_vector:
        for word, count in document.items():
            if count > 0:
                idf[word] += 1
 
    for word, v in idf.items():
        idf[word] = log(n / 1 + float(v))
    return idf

In [35]:

def compute_tf_idf(tf, idf):
    tf_idf = dict.fromkeys(tf.keys(), 0)
    for word, v in tf.items():
        tf_idf[word] = v * idf[word]
    return tf_idf

# Tính độ lệch chuẩn theo paper

In [36]:
num_of_document = len(data)  
_CONTENT_HEADER = 'description'
_PUBLIC_DATE_HEADER = 'public_date'

copus = data[_CONTENT_HEADER].head(num_of_document)

all_document = copus.values.tolist()

cv = CountVectorizer(all_document, tokenizer=vi_term_tokenize, lowercase=False)
word_count_vector = cv.fit_transform(all_document)

idf_list = []

data['time'] = pd.to_datetime(data[_PUBLIC_DATE_HEADER], format="%Y-%m-%d %H:%M:%S")

input_data = data.head(num_of_document).set_index('time')

for _, document_group in input_data.groupby(pd.Grouper(freq='2D')): 
    
    document_group_word_dict = []
    all_features = cv.get_feature_names()
   
    document_group_content = document_group[_CONTENT_HEADER].values.tolist()
    for document in document_group_content:
        document_group_word_dict.append(dict.fromkeys(all_features, 0))

    count = 0
    for document_token_vector in document_group_word_dict:
        document_tokens = vi_term_tokenize(document_group_content[count])
        
        for token in document_tokens:
            if token in document_token_vector: 
                document_token_vector[token] += 1
        count = count + 1
   
    idf = compute_idf(document_group_word_dict)
    idf_list.append(idf)
print("Số nhóm : ", len(idf_list))
ajust_time = {}

for token in cv.get_feature_names():
    idf_vals = []
    for idf_dict in idf_list:
        idf_vals.append(idf_dict[token])
    
   
    SD = np.std(idf_vals, dtype=np.float64, ddof=0) 
  
    mean = np.mean(idf_vals)
   
    above = [x for x in idf_vals if x > mean]
   
    SD_Above = np.std(above, dtype=np.float64, ddof=0)
    
    below = [x for x in idf_vals if x < mean]
   
    SD_Below = np.std(below, dtype=np.float64, ddof=0)

    ajust_time[token] = (SD_Above + 0.001) / (SD_Below + 0.001) 

Số nhóm :  4


# In ra danh sách từ khoá sau khi sắp xếp để kiểm tra

In [38]:
import operator
sorted_x = sorted(ajust_time.items(), key=operator.itemgetter(1), reverse = True)
for key, value in sorted_x:
    print(key, '=>', value)

lựa_chọn => 153.69082477559093
chuyên_gia => 140.35670123451027
vận => 126.65721414045305
hiệu_quả => 126.65721414045305
con_giáp => 126.65721414045305
thành_công => 126.65721414045305
bất_động_sản => 126.65721414045305
thiết_bị => 126.65721414045305
quen => 126.65721414045305
khủng_hoảng => 126.65721414045305
giao_dịch => 126.65721414045305
tương_lai => 126.65721414045305
xóa => 112.57177565710485
lao_lý => 112.57177565710485
bóng => 112.57177565710485
vũ_khí => 112.57177565710485
biến => 112.57177565710485
cứu => 112.57177565710485
Con_cái => 112.57177565710485
phép => 112.57177565710485
danh_hiệu => 112.57177565710485
mét => 112.57177565710485
dự_án => 112.57177565710485
nghi => 112.57177565710485
thoát => 112.57177565710485
đắt => 112.57177565710485
Trung_niên => 112.57177565710485
chăm_sóc => 112.57177565710485
chín_chắn => 112.57177565710485
quý => 112.57177565710485
thành_tựu => 112.57177565710485
OYO => 112.57177565710485
Harvard => 112.57177565710485
thức_tỉnh => 112.571775657

đời => 2.2054851441826404
trải => 2.097079638027961
đối_thủ => 2.097079638027961
Đấy => 2.097079638027961
anh_chị => 2.097079638027961
cơ_thể => 2.097079638027961
Đề_xuất => 2.097079638027961
Malaysia => 2.097079638027961
mắt_kính => 2.097079638027961
series => 2.097079638027961
danh_họa => 2.097079638027961
cảng => 2.097079638027961
bài_tập => 2.097079638027961
tháng_chạp => 2.097079638027961
đội_tuyển => 2.097079638027961
mua => 2.097079638027961
nhập_viện => 2.097079638027961
nhẹ => 2.097079638027961
quan_tài => 2.097079638027961
động_cơ => 2.097079638027961
vô_địch => 2.097079638027961
ACV => 2.097079638027961
kid => 2.097079638027961
luật => 2.097079638027961
chợ => 2.097079638027961
công_chiếu => 2.097079638027961
fan => 2.097079638027961
quyền_lực => 2.097079638027961
rượu_bia => 2.097079638027961
nói_xấu => 2.097079638027961
rich => 2.097079638027961
tích_cực => 2.097079638027961
Đại_sứ_quán => 2.097079638027961
đứng => 2.097079638027961
Nguyễn_Tiế => 2.097079638027961
đừng => 